# WP8 — Certified Robustness for Value Learning
**Prometheus v0.97**

This notebook demonstrates the randomised-smoothing certification layer
added to the ``ValueLearningAgent`` in WP8:

1. **Motivation** — why L₂ adversarial perturbations can flip preferences
2. **Randomised smoothing** — the Cohen et al. (2019) framework adapted to Bradley-Terry
3. **Clopper-Pearson lower bound** — statistical confidence for the certificate
4. **Sigma sweep** — trading accuracy for certified radius
5. **Epsilon sweep** — certified fraction vs perturbation budget
6. **Adversarial attack** — certified preferences hold under actual L₂ attack
7. **CertifiedDataValidator** — WP3 DataValidator + certified-radius check


In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    os.system('pip install scipy -q')
    os.system('git clone https://github.com/prometheus-ai/Prometheus_v0_PoC /content/Prometheus_v0_PoC 2>/dev/null || true')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.abspath('..'))

import warnings; warnings.filterwarnings('ignore')
print('Environment ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from prometheus.value_learning import ValueLearningAgent
from prometheus.certified_robustness import (
    SmoothedValueLearner, CertifiedDataValidator,
    _clopper_pearson_lower, _certified_radius
)
from benchmarks.certified_robustness_benchmark import (
    CertifiedRobustnessBenchmark, _make_pairs, _trained_agent, N_FEATS
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Train base agent
agent = _trained_agent(n_pairs=80, seed=0)
print(f'Trained agent: {agent}')

---
## 1 — Motivation: Adversarial Perturbations Flip Preferences

In [ ]:
rng  = np.random.default_rng(42)
phi1 = rng.uniform(0.4, 0.6, N_FEATS)
phi2 = rng.uniform(0.4, 0.6, N_FEATS)

# Ensure phi1 is preferred by the agent
if agent.get_reward(phi2) > agent.get_reward(phi1):
    phi1, phi2 = phi2, phi1

print(f'Base reward  phi1: {agent.get_reward(phi1):.4f}')
print(f'Base reward  phi2: {agent.get_reward(phi2):.4f}')
print(f'Base preference phi1 ≻ phi2: {agent.get_reward(phi1) > agent.get_reward(phi2)}')

# Find an adversarial direction: maximise phi2's reward
w         = agent.weights
adv_dir   = -w / (np.linalg.norm(w) + 1e-12)

print('\nEffect of adversarial perturbation on phi1:')
for eps in [0.0, 0.05, 0.10, 0.20, 0.50]:
    phi1_adv  = phi1 + eps * adv_dir
    r1 = agent.get_reward(phi1_adv)
    r2 = agent.get_reward(phi2)
    flipped = r2 > r1
    print(f'  ε={eps:.2f}: reward={r1:.4f} vs {r2:.4f}  flipped={flipped}')

---
## 2 — Randomised Smoothing: Certify a Preference

In [ ]:
smoother = SmoothedValueLearner(agent, sigma=0.1, n_samples=1000, seed=42)

# Strongly preferred pair
w    = agent.weights
phi1_strong = np.abs(w) * 2.0
phi2_strong = np.zeros(N_FEATS)

cr = smoother.certify(phi1_strong, phi2_strong)
print(cr.summary())
print(f'\nPrefers phi1: {cr.prefers_phi1}')
print(f'p estimate:   {cr.p_estimate:.4f}')
print(f'p lower:      {cr.p_lower_bound:.4f}  (99% confidence)')
print(f'Cert. radius: {cr.certified_radius:.4f}')
print(f'Abstain:      {cr.abstain}')
print(f'Time:         {cr.certify_time_s*1000:.1f} ms')

---
## 3 — Clopper-Pearson Confidence and the Certificate

In [ ]:
N = 1000
k_values = range(500, 1001, 10)
p_lowers = [_clopper_pearson_lower(k, N, 0.01) for k in k_values]  # 99% confidence
radii    = [_certified_radius(pl, sigma=0.1) for pl in p_lowers]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot([k/N for k in k_values], p_lowers, 'b-', linewidth=2)
ax.axhline(0.5, color='r', linestyle='--', alpha=0.7, label='p=0.5 (no cert.)')
ax.set_xlabel('p̂ = k/N (fraction voting phi1 ≻ phi2)')
ax.set_ylabel('p_lower (99% C.P. bound)')
ax.set_title('Clopper-Pearson Lower Bound\nvs Monte Carlo Vote Fraction', fontweight='bold')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax2 = axes[1]
ax2.plot([k/N for k in k_values], radii, 'g-', linewidth=2)
ax2.axhline(0, color='r', linestyle='--', alpha=0.7, label='R=0 (abstain)')
ax2.set_xlabel('p̂ = k/N')
ax2.set_ylabel('Certified radius R = σ·Φ⁻¹(p_lower)')
ax2.set_title('Certified L₂ Radius vs Vote Fraction\n(σ=0.1)', fontweight='bold')
ax2.legend(); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## 4 — Benchmark: Sigma Sweep and Epsilon Sweep

In [ ]:
bench   = CertifiedRobustnessBenchmark(n_pairs=40, n_samples=400, seed=42)
results = bench.run_all()
print(bench.summary_table(results))

In [ ]:
sig_r = next(r for r in results if r.scenario == 'sigma_sweep')
eps_r = next(r for r in results if r.scenario == 'epsilon_sweep')
adv_r = next(r for r in results if r.scenario == 'adversarial')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sigma sweep
ax = axes[0]
ax2_twin = ax.twinx()
ax.plot(sig_r.param_values, [f*100 for f in sig_r.certified_fractions],
        'b-o', linewidth=2, label='Certified fraction (%)')
ax2_twin.plot(sig_r.param_values, [a*100 for a in sig_r.preference_accuracies],
              'r--s', linewidth=2, label='Preference accuracy (%)')
ax.set_xlabel('σ (noise level)')
ax.set_ylabel('Certified fraction (%)', color='blue')
ax2_twin.set_ylabel('Accuracy (%)', color='red')
ax.set_title('Sigma Sweep (ε=0.1)', fontweight='bold')
ax.legend(loc='upper left'); ax2_twin.legend(loc='upper right')
ax.spines['top'].set_visible(False)

# Epsilon sweep
ax = axes[1]
ax.plot(eps_r.param_values, [f*100 for f in eps_r.certified_fractions],
        'g-o', linewidth=2)
ax.set_xlabel('ε (perturbation budget)')
ax.set_ylabel('Certified fraction (%)')
ax.set_title('Epsilon Sweep (σ=0.1)', fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Adversarial: certified accuracy under attack
ax = axes[2]
valid = [(pv, ca) for pv, ca in
         zip(adv_r.param_values, adv_r.certified_accuracy_attacks)
         if ca == ca]   # filter NaN
if valid:
    pvs, cas = zip(*valid)
    ax.plot(pvs, [c*100 for c in cas], 'purple', marker='D', linewidth=2,
            label='Cert. acc. under attack')
    ax.axhline(100, color='gray', linestyle='--', alpha=0.5, label='100% ideal')
ax.set_xlabel('ε (cert. threshold)')
ax.set_ylabel('Certified accuracy under L₂ attack (%)')
ax.set_title(f'Adversarial Accuracy\n(ε_attack=0.05, σ=0.1)', fontweight='bold')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig.suptitle('Certified Robustness Benchmark — Prometheus v0.97 (WP8)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5 — CertifiedDataValidator

In [ ]:
smoother = SmoothedValueLearner(agent, sigma=0.1, n_samples=500, seed=0)
cdv      = CertifiedDataValidator(n_features=N_FEATS, smoother=smoother)

w     = agent.weights
phi1  = np.abs(w) * 2.0    # strongly preferred
phi2  = np.zeros(N_FEATS)  # unpreferred reference

print('Testing CertifiedDataValidator:')
for eps in [0.001, 0.05, 0.10, 0.50]:
    safe, radius = cdv.validate_and_certify(phi1, epsilon=eps, reference=phi2)
    print(f'  ε={eps:.3f}: safe={safe}, certified_radius={radius:.4f}')

# NaN injection
print('\nNaN injection:')
nan_arr = np.array([float('nan')] * N_FEATS)
safe, radius = cdv.validate_and_certify(nan_arr, epsilon=0.1, reference=phi2)
print(f'  safe={safe}, radius={radius}  (rejected by DataValidator)')

---
## Summary

| Property | Mechanism | Status |
|----------|-----------|--------|
| Preference stability certificate | Randomised smoothing (Cohen et al. 2019) | Implemented |
| Statistical confidence | Clopper-Pearson 99% lower bound | Implemented |
| Certified radius | R = σ · Φ⁻¹(p_lower) | Implemented |
| Abstention on unclear preference | p_lower ≤ 0.5 → abstain | Implemented |
| NaN/Inf protection | WP3 DataValidator integrated | Implemented |
| Adversarial accuracy | Certified pairs hold under L₂ attack | Demonstrated |

**Test coverage**: 50 tests, all passing (`pytest tests/test_certified_robustness.py -v`)

**Key trade-off**: Larger σ → larger certified radius but lower preference accuracy.
Recommended operating point: σ = 0.1, ε = 0.05 for the 5-feature IRL setting.

**Files**:
- `prometheus/certified_robustness.py` — SmoothedValueLearner, CertifiedDataValidator
- `benchmarks/certified_robustness_benchmark.py` — sigma/epsilon/adversarial benchmark
- `tests/test_certified_robustness.py` — 50-test suite
